In [1]:
# !pip install torch
# !pip install transformers
# !pip install datasets
# !pip install trl
# !pip install peft
# !pip install accelerate
# !pip install bitsandbytes
# !pip install scikit-learn
# !pip install numpy
# !pip install wandb
# !pip install tqdm

In [2]:
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TrainerCallback
from datasets import Dataset
import os
import json
import re
import wandb
import pandas as pd

import trl
from trl import GRPOConfig, GRPOTrainer
print(f"trl version: {trl.__version__}")

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

trl version: 1.4.0


# 1. Model Selection and Loading

In [3]:
# Options for small models
model_options = {
    "qwen-0.5b-instruct": "Qwen/Qwen2.5-0.5B-Instruct",
    "qwen-1.5b-instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    "gpt2": "gpt2",  # 124M parameters
    "gpt2-medium": "gpt2-medium",  # 355M parameters
    "opt-125m": "facebook/opt-125m",
    "bloom-560m": "bigscience/bloom-560m"
}

# Choose a model
MODEL_CHOICE = "gpt2"
model_name = model_options[MODEL_CHOICE]

print(f"Selected model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model with optimal settings for limited resources
device_map = "auto"
if torch.cuda.is_available():
    print("Using CUDA")
elif torch.backends.mps.is_available():
    print("CUDA unavailable, using MPS for Mac")
    device_map = {"": "mps"}
else:
    print("CUDA and MPS unavailable, using CPU")
    device_map = {"": "cpu"}

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    torch_dtype=torch.bfloat16,
    device_map=device_map,
    low_cpu_mem_usage=True
)

Selected model: gpt2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Using CUDA


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# 2. Dataset Creation

In [4]:
def generate_math_problem():
    """Generates a random math problem."""
    operations = ["+", "-", "*", "/"]
    op = random.choice(operations)

    if op == "+":
        a = random.randint(1, 100)
        b = random.randint(1, 100)
        answer = a + b
        prompt = f"What is {a} plus {b}?"
    elif op == "-":
        a = random.randint(1, 100)
        b = random.randint(1, min(a, 100))  # To ensure a positive result
        answer = a - b
        prompt = f"What is {a} minus {b}?"
    elif op == "*":
        a = random.randint(1, 20)
        b = random.randint(1, 20)
        answer = a * b
        prompt = f"What is {a} multiplied by {b}?"
    else:  # "/"
        b = random.randint(1, 10)
        a = b * random.randint(1, 10)  # To ensure an integer result
        answer = a // b
        prompt = f"What is {a} divided by {b}?"

    return prompt, answer

# Create training and validation datasets
def create_datasets(train_size=200, val_size=50):
    """Creates training and validation datasets."""
    all_data = []

    # Generate data
    for _ in range(train_size + val_size):
        prompt, answer = generate_math_problem()
        all_data.append({"prompt": prompt, "answer": answer})

    random.shuffle(all_data)

    train_data = all_data[:train_size]
    val_data = all_data[train_size:]

    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)

    return train_dataset, val_dataset

In [5]:
train_dataset, val_dataset = create_datasets(train_size=200, val_size=50)

def format_prompt(problem):
    return (
        "Solve the arithmetic problem.\n"
        "Output exactly one line in the format: Answer: <integer>\n"
        f"Problem: {problem}\n"
    )

train_dataset = train_dataset.map(lambda x: {
    "prompt": format_prompt(x["prompt"]),
    "answer": x["answer"]
})

val_dataset = val_dataset.map(lambda x: {
    "prompt": format_prompt(x["prompt"]),
    "answer": x["answer"]
})

print("Examples from the training dataset:")
for i in range(3):
    print(f"Example {i+1}: {train_dataset[i]}")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Examples from the training dataset:
Example 1: {'prompt': 'Solve the arithmetic problem.\nOutput exactly one line in the format: Answer: <integer>\nProblem: What is 20 plus 81?\n', 'answer': 101}
Example 2: {'prompt': 'Solve the arithmetic problem.\nOutput exactly one line in the format: Answer: <integer>\nProblem: What is 72 plus 26?\n', 'answer': 98}
Example 3: {'prompt': 'Solve the arithmetic problem.\nOutput exactly one line in the format: Answer: <integer>\nProblem: What is 54 plus 53?\n', 'answer': 107}


# 3. Base Model Testing

In [6]:
# Test the base model on a few examples
print("\nTesting the base model:")

for i in range(5):
    prompt = train_dataset[i]["prompt"]
    expected = train_dataset[i]["answer"]

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=6,
        min_new_tokens=1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    print("="*50)
    print(f"\nPrompt:\n{prompt}")
    print(f"Expected answer: {expected}")
    print(f"Generated completion: {response}")
    print("="*50)
    print("\n")


Testing the base model:

Prompt:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 20 plus 81?

Expected answer: 101
Generated completion: Solution: Answer: <integer



Prompt:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 72 plus 26?

Expected answer: 98
Generated completion: Solution: Answer: <integer



Prompt:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 54 plus 53?

Expected answer: 107
Generated completion: Answer: <integer>




Prompt:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 92 plus 37?

Expected answer: 129
Generated completion: Answer: <integer>




Prompt:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 69 minus 32?

Expected answer: 37
Generated completion: Answer: <integer>





# 4. Reward Function

In [10]:
def reward_function(prompts, completions, **kwargs):
    """
    Reward function for evaluating model responses.

    Parameters:
    - prompts: list of problems
    - completions: list of model responses
    - kwargs: additional arguments that GRPOTrainer may pass

    Returns:
    - list of rewards for each response
    """
    rewards = []

    ground_truth = kwargs.get('answer', [None] * len(prompts))

    for prompt, completion, gt in zip(prompts, completions, ground_truth):
        reward = 0.0

        # Extract numerical answer
        match = re.search(r'^\s*Answer:\s*(\d+)', completion)

        format_correct = bool(match)

        if format_correct:
            reward += 1.0

            # Check answer correctness if ground_truth is available
            if gt is not None:
                try:
                    provided_answer = int(match.group(1))

                    after_number = completion[match.end():].strip() # Penalty for extra tokens after answer
                    if after_number:
                        extra_tokens = len(after_number.split())

                        reward -= 1.0 * extra_tokens

                    # Differentiated reward based on closeness to correct answer
                    if provided_answer == gt:
                        reward += 2.0  # Full reward for correct answer
                    elif abs(provided_answer - gt) <= 5:
                        reward += 1.0  # Partial reward for close answer
                    elif abs(provided_answer - gt) <= 10:
                        reward += 0.5  # Small reward for not too distant answer
                except:
                    pass

        rewards.append(reward)

    return rewards

# 5. GRPO Trainer Configuration

In [11]:
# GRPO configuration with optimal parameters for small models
grpo_config = GRPOConfig(
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_generations=4,
    # beta=0.1,  # Target KL divergence
    seed=SEED,
    scale_rewards=True,  # Reward scaling
    output_dir="./grpo_checkpoint",  # Directory for checkpoint saving
    logging_steps=10,  # Log every 10 steps
    save_strategy="epoch",  # Save model at the end of each epoch
    eval_strategy="epoch",  # Evaluate model at the end of each epoch
    num_train_epochs=3,
    max_completion_length=4,
    report_to=["wandb"],  # Enable wandb reports
    log_completions=True,  # Enable text example logging
    # wandb_log_unique_prompts=True  # Log only unique prompts to save space
)

# GRPO trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    reward_funcs=[reward_function],
)

# 6. Data Preparation

In [12]:
def prepare_dataset_for_grpo(dataset):
    """Estimates initial reward before GRPO training."""
    problems = [item["prompt"] for item in dataset]
    expected_answers = [item["answer"] for item in dataset]

    initial_responses = []
    print("Getting initial model responses...")

    model.eval()

    with torch.no_grad():
        for problem in tqdm(problems[:50]):
            inputs = tokenizer(problem, return_tensors="pt").to(model.device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=4,
                min_new_tokens=1,
                pad_token_id=tokenizer.eos_token_id
            )
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            initial_responses.append(response)

    initial_rewards = reward_function(
        problems[:50],
        initial_responses,
        answer=expected_answers[:50]
    )
    avg_initial_reward = sum(initial_rewards) / len(initial_rewards)

    print(f"Average initial reward on sample: {avg_initial_reward:.2f}")


def evaluate_model_for_wandb(model, tokenizer, test_problems=None, epoch=0):
    """
    Evaluates the model on test examples and logs results to wandb.

    Parameters:
    - model: trained model
    - tokenizer: tokenizer
    - test_problems: list of test problems (if None, predefined ones are used)
    - epoch: epoch number
    """
    if test_problems is None:
        test_problems = [
            "What is 15 plus 27?",
            "What is 42 minus 18?",
            "What is 7 multiplied by 8?",
            "What is 45 divided by 9?"
        ]

    results = []

    for problem in test_problems:
        formatted_prompt = format_prompt(problem)

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            inputs["input_ids"],
            max_new_tokens=4,
            min_new_tokens=1,
            pad_token_id=tokenizer.eos_token_id
        )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        match = re.search(r'Answer:\s*(\d+)', response)
        answer = match.group(1) if match else "Invalid format"

        results.append({
            "Problem": problem,
            "Model Response": response,
            "Numerical Answer": answer
        })

    if wandb.run is not None:
        wandb.log({
            "model_samples/epoch": epoch,
            "model_samples/examples": wandb.Table(
                dataframe=pd.DataFrame(results)
            )
        })

    return results


class WandbEvalCallback(TrainerCallback):
    """Callback for logging generation results to wandb after each epoch"""

    def __init__(self, model, tokenizer, test_problems=None):
        self.model = model
        self.tokenizer = tokenizer
        self.test_problems = test_problems

    def on_epoch_end(self, args, state, control, **kwargs):
        evaluate_model_for_wandb(
            self.model,
            self.tokenizer,
            self.test_problems,
            state.epoch
        )
        return control

if "wandb" in grpo_config.report_to:
    test_problems = [
        "What is 15 plus 27?",
        "What is 42 minus 18?",
        "What is 7 multiplied by 8?",
        "What is 45 divided by 9?",
        "What is 33 plus 44?",
        "What is 99 minus 34?",
        "What is 12 multiplied by 5?",
        "What is 72 divided by 8?"
    ]
    trainer.add_callback(WandbEvalCallback(model, tokenizer, test_problems))

In [13]:
train_subset = train_dataset.select(range(50))
prepare_dataset_for_grpo(train_subset)

Getting initial model responses...


  0%|          | 0/50 [00:00<?, ?it/s]

Average initial reward on sample: 0.00


# 7. Training

In [14]:
print("\nStarting GRPO training...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.



Starting GRPO training...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


...

╭─────────────────────────────────────────────── Step 150 ────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                   ┃ Completion ┃ reward_function ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Solve the arithmetic problem.                            │ Answer: 17 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 70 minus 17?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 17 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 70 minus 17?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 17 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 70 minus 17?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 17 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 70 minus 17?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 28 │            1.50 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 47 minus 28?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 28 │            1.50 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 47 minus 28?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 28 │            1.50 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 47 minus 28?                            │            │                 │           │ │
│ │ 

wandb: WARNING URL not available in offline run


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

╭─────────────────────────────────────────────── Step 150 ────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                                   ┃ Completion ┃ reward_function ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Solve the arithmetic problem.                            │ Answer: 17 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 70 minus 17?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 17 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 70 minus 17?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 17 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 70 minus 17?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 17 │            1.00 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 70 minus 17?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 28 │            1.50 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 47 minus 28?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 28 │            1.50 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 47 minus 28?                            │            │                 │           │ │
│ │                                                          │            │                 │           │ │
│ ├──────────────────────────────────────────────────────────┼────────────┼─────────────────┼───────────┤ │
│ │ Solve the arithmetic problem.                            │ Answer: 28 │            1.50 │      0.00 │ │
│ │ Output exactly one line in the format: Answer: <integer> │            │                 │           │ │
│ │ Problem: What is 47 minus 28?                            │            │                 │           │ │
│ │ 

TrainOutput(global_step=150, training_loss=6.953875223795573e-10, metrics={'train_runtime': 117.7795, 'train_samples_per_second': 5.094, 'train_steps_per_second': 1.274, 'total_flos': 0.0, 'train_loss': 6.953875223795573e-10})

# 8. Final Comparison Before/After

In [15]:
print("\nFinal comparison of model before and after training:")

# Load original model (before training)
original_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map=device_map,
    low_cpu_mem_usage=True
)

test_problems = [
    "What is 15 plus 27?",
    "What is 42 minus 18?",
    "What is 7 multiplied by 8?",
    "What is 45 divided by 9?"
]

print("\nComparison of responses on new examples:")

for problem in test_problems:

    prompt = format_prompt(problem)

    # Original model
    original_inputs = tokenizer(prompt, return_tensors="pt").to(original_model.device)

    original_outputs = original_model.generate(
        **original_inputs,
        max_new_tokens=4,
        min_new_tokens=1,
        pad_token_id=tokenizer.eos_token_id
    )

    original_response = tokenizer.decode(
        original_outputs[0],
        skip_special_tokens=True
    )

    # Trained model
    trained_inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)

    trained_outputs = trainer.model.generate(
        **trained_inputs,
        max_new_tokens=4,
        min_new_tokens=1,
        pad_token_id=tokenizer.eos_token_id
    )

    trained_response = tokenizer.decode(
        trained_outputs[0],
        skip_special_tokens=True
    )

    print("=" * 60)
    print(f"Problem: {problem}")
    print(f"\nOriginal model:\n{original_response}")
    print(f"\nTrained model:\n{trained_response}")
    print("=" * 60)


Final comparison of model before and after training:


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Comparison of responses on new examples:
Problem: What is 15 plus 27?

Original model:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 15 plus 27?
Answer: <integer

Trained model:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 15 plus 27?
Answer: 27

Problem: What is 42 minus 18?

Original model:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 42 minus 18?
Solution: Answer:

Trained model:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 42 minus 18?
Answer: 18

Problem: What is 7 multiplied by 8?

Original model:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: What is 7 multiplied by 8?
Solution: Answer:

Trained model:
Solve the arithmetic problem.
Output exactly one line in the format: Answer: <integer>
Problem: Wh